## 1. Loading and Inspecting the Dataset

In this step, I will load the movie ratings dataset into Google Colab and take a look at its contents. I will display the first 10 records, check the number of rows and columns, and examine the data types. I will also check for missing values and duplicate records to understand the quality of the data before cleaning it.


In [2]:
# Import the required libraries
import pandas as pd
from google.colab import files

# Upload the dataset
uploaded = files.upload()

# Load the uploaded CSV file
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

# Display the first 10 records
print("First 10 Records:")
display(df.head(10))

# Display dataset dimensions
print("\nDataset Dimensions:")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Display data types
print("\nData Types:")
print(df.dtypes)

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check for duplicate records
print("\nNumber of Duplicate Records:")
print(df.duplicated().sum())

Saving DSA_4060_Week2_Movie_Ratings.csv to DSA_4060_Week2_Movie_Ratings (1).csv
First 10 Records:


,user_id,movie_id,movie_title,rating,timestamp
0,U001,M001,Nairobi Nights,3,2026-01-01T08:00:00Z
1,U004,M001,Nairobi Nights,4,2026-01-06T09:00:00Z
2,U007,M001,Nairobi Nights,2,2026-01-11T10:00:00Z
3,U010,M001,Nairobi Nights,2,2026-01-16T11:00:00Z
4,U013,M001,Nairobi Nights,4,2026-01-21T12:00:00Z
5,U016,M001,Nairobi Nights,3,2026-01-26T13:00:00Z
6,U019,M001,Nairobi Nights,3,2026-01-31T14:00:00Z
7,U022,M001,Nairobi Nights,2,2026-02-05T15:00:00Z
8,U025,M001,Nairobi Nights,2,2026-02-10T16:00:00Z
9,U028,M001,Nairobi Nights,4,2026-02-15T17:00:00Z



Dataset Dimensions:
Number of rows: 509
Number of columns: 5

Data Types:
user_id        object
movie_id       object
movie_title    object
rating          int64
timestamp      object
dtype: object

Missing Values:
user_id        2
movie_id       1
movie_title    1
rating         0
timestamp      0
dtype: int64

Number of Duplicate Records:
3


## 2. Data Cleaning

Before analyzing the ratings, I need to clean the dataset to ensure that the results are reliable. In this step, I will remove duplicate records and entries with missing user or movie IDs. I will also check whether the ratings fall within the expected range of 1 to 5. Finally, I will verify the dataset after cleaning to see how many records remain.


In [3]:
# Make a copy of the original dataset
df_clean = df.copy()

# Check the number of records before cleaning
print("Records before cleaning:", len(df_clean))

# Remove duplicate records
df_clean = df_clean.drop_duplicates()

# Remove records with missing user IDs or movie IDs
df_clean = df_clean.dropna(subset=['user_id', 'movie_id'])

# Check the expected rating scale (1 to 5)
print("\nUnique ratings:")
print(sorted(df_clean['rating'].unique()))

# Identify ratings outside the expected range
invalid_ratings = df_clean[
    (df_clean['rating'] < 1) | (df_clean['rating'] > 5)
]

print("\nNumber of invalid ratings:", len(invalid_ratings))

# Display the cleaned dataset information
print("\nRecords after cleaning:", len(df_clean))

print("\nMissing values after cleaning:")
print(df_clean.isnull().sum())

print("\nRemaining duplicate records:", df_clean.duplicated().sum())

# Display the first 10 records of the cleaned dataset
print("\nFirst 10 records after cleaning:")
display(df_clean.head(10))

Records before cleaning: 509

Unique ratings:
[np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Number of invalid ratings: 3

Records after cleaning: 503

Missing values after cleaning:
user_id        0
movie_id       0
movie_title    0
rating         0
timestamp      0
dtype: int64

Remaining duplicate records: 0

First 10 records after cleaning:


,user_id,movie_id,movie_title,rating,timestamp
0,U001,M001,Nairobi Nights,3,2026-01-01T08:00:00Z
1,U004,M001,Nairobi Nights,4,2026-01-06T09:00:00Z
2,U007,M001,Nairobi Nights,2,2026-01-11T10:00:00Z
3,U010,M001,Nairobi Nights,2,2026-01-16T11:00:00Z
4,U013,M001,Nairobi Nights,4,2026-01-21T12:00:00Z
5,U016,M001,Nairobi Nights,3,2026-01-26T13:00:00Z
6,U019,M001,Nairobi Nights,3,2026-01-31T14:00:00Z
7,U022,M001,Nairobi Nights,2,2026-02-05T15:00:00Z
8,U025,M001,Nairobi Nights,2,2026-02-10T16:00:00Z
9,U028,M001,Nairobi Nights,4,2026-02-15T17:00:00Z


### Handling Invalid Ratings

After checking the ratings, I found three values that fall outside the expected scale of 1 to 5. I will remove these records to ensure that the remaining data contains only valid ratings. I will then verify the dataset to confirm that the cleaning process is complete.


In [4]:
# Remove ratings outside the expected range of 1 to 5
df_clean = df_clean[
    (df_clean['rating'] >= 1) & (df_clean['rating'] <= 5)
].copy()

# Display the number of records remaining
print("Records after removing invalid ratings:", len(df_clean))

# Confirm that all ratings are valid
print("\nValid ratings:")
print(sorted(df_clean['rating'].unique()))

# Final data quality check
print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nDuplicate records:", df_clean.duplicated().sum())

Records after removing invalid ratings: 500

Valid ratings:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Missing values:
user_id        0
movie_id       0
movie_title    0
rating         0
timestamp      0
dtype: int64

Duplicate records: 0


### Checking for Repeated User-Movie Pairs

While looking at the user-item matrix, I noticed some cells contained values like 2.33 instead of whole ratings. This suggests that some users rated the same movie more than once. Before going further, I will check how many user-movie pairs appear more than once, since each user should only have one rating per movie.

In [10]:
# Find rows where the same user rated the same movie more than once
pair_duplicates = df_clean.duplicated(subset=['user_id', 'movie_id'], keep=False)

print("Rows involved in repeated user-movie pairs:", pair_duplicates.sum())
print("Extra rows (beyond the first rating of each pair):",
      df_clean.duplicated(subset=['user_id', 'movie_id']).sum())
print("Unique user-movie pairs:",
      df_clean[['user_id', 'movie_id']].drop_duplicates().shape[0])

# Show some examples, sorted so repeated ratings sit together
display(
    df_clean[pair_duplicates]
    .sort_values(['user_id', 'movie_id', 'timestamp'])
    .head(15)
)

Rows involved in repeated user-movie pairs: 127
Extra rows (beyond the first rating of each pair): 70
Unique user-movie pairs: 430


,user_id,movie_id,movie_title,rating,timestamp
355,U002,M004,Coastal Dreams,3,2026-03-11T09:00:00Z
46,U002,M004,Coastal Dreams,3,2026-03-19T08:00:00Z
402,U002,M004,Coastal Dreams,3,2026-06-07T12:00:00Z
321,U002,M004,Coastal Dreams,5,2026-07-13T08:00:00Z
289,U004,M005,Mountain Echoes,2,2026-02-20T13:00:00Z
455,U004,M005,Mountain Echoes,4,2026-06-12T11:00:00Z
359,U004,M019,Future Africa,5,2026-02-27T10:00:00Z
225,U004,M019,Future Africa,4,2026-07-27T17:00:00Z
254,U004,M022,Wings of Hope,4,2026-07-19T10:00:00Z
454,U004,M022,Wings of Hope,2,2026-07-29T19:00:00Z


### Keeping the Latest Rating per User-Movie Pair

The check showed 70 extra rows where a user rated the same movie again at a later time, and the ratings often differed. Since each user should have one rating per movie, I will keep only the most recent rating for each pair, because it best reflects what the user currently thinks of the movie. This leaves 430 unique user-movie pairs.

In [11]:
# Convert timestamp to a proper datetime so sorting is reliable
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])

# Sort by time, then keep the latest rating for each user-movie pair
df_clean = (
    df_clean
    .sort_values('timestamp')
    .drop_duplicates(subset=['user_id', 'movie_id'], keep='last')
    .sort_values(['user_id', 'movie_id'])
    .reset_index(drop=True)
)

# Confirm the result
print("Records after keeping latest rating per pair:", len(df_clean))
print("Remaining repeated user-movie pairs:",
      df_clean.duplicated(subset=['user_id', 'movie_id']).sum())
print("Unique users:", df_clean['user_id'].nunique())
print("Unique movies:", df_clean['movie_id'].nunique())

Records after keeping latest rating per pair: 430
Remaining repeated user-movie pairs: 0
Unique users: 50
Unique movies: 24


### Checking ID and Title Consistency

Before recalculating the statistics, I will check that the user IDs, movie IDs and movie titles are consistent. Extra spaces, mixed letter cases or a movie with more than one title could make the same user or movie count as two, which would affect the results.

In [12]:
# Check user and movie IDs for stray spaces or case differences
for col in ['user_id', 'movie_id']:
    cleaned = df_clean[col].str.strip().str.upper()
    print(f"{col}: {df_clean[col].nunique()} unique as stored, "
          f"{cleaned.nunique()} unique after trimming and uppercasing")

# Check that IDs follow the expected pattern (U001, M001, ...)
print("\nUser IDs not matching pattern:",
      (~df_clean['user_id'].str.match(r'^U\d{3}$')).sum())
print("Movie IDs not matching pattern:",
      (~df_clean['movie_id'].str.match(r'^M\d{3}$')).sum())

# Check that each movie_id has exactly one title, and each title one movie_id
print("\nMax titles per movie_id:",
      df_clean.groupby('movie_id')['movie_title'].nunique().max())
print("Max movie_ids per title:",
      df_clean.groupby('movie_title')['movie_id'].nunique().max())

user_id: 50 unique as stored, 50 unique after trimming and uppercasing
movie_id: 24 unique as stored, 24 unique after trimming and uppercasing

User IDs not matching pattern: 0
Movie IDs not matching pattern: 0

Max titles per movie_id: 1
Max movie_ids per title: 1


**Interpretation:**

 The checks show that the data is consistent. The number of unique user IDs (50) and movie IDs (24) stayed the same after trimming spaces and converting to uppercase, so there are no hidden variations of the same ID. All IDs follow the expected format, and each movie ID has exactly one title and each title has exactly one movie ID. This means the counts I calculate next will not be inflated by inconsistent entries.

## 3. Exploratory Data Analysis

In this step, I will explore the cleaned dataset to understand how users have rated the movies. I will calculate the total number of ratings, the number of unique users and movies, and the average rating. I will also identify the most active user and the movie that has received the most ratings.


In [13]:
# Calculate the main statistics

# Total number of ratings
total_ratings = len(df_clean)

# Number of unique users
unique_users = df_clean['user_id'].nunique()

# Number of unique movies
unique_movies = df_clean['movie_id'].nunique()

# Most active user
user_counts = df_clean['user_id'].value_counts()
most_active_user = user_counts.idxmax()
most_active_user_count = user_counts.max()

# Most-rated movie
movie_counts = df_clean['movie_id'].value_counts()
most_rated_movie_id = movie_counts.idxmax()
most_rated_movie_count = movie_counts.max()

# Get the movie title
most_rated_movie_title = df_clean.loc[
    df_clean['movie_id'] == most_rated_movie_id, 'movie_title'
].iloc[0]

# Average rating
average_rating = df_clean['rating'].mean()

# Display the results
print("EXPLORATORY DATA ANALYSIS")
print("-" * 40)

print("Total number of ratings:", total_ratings)
print("Number of unique users:", unique_users)
print("Number of unique movies:", unique_movies)

print("\nMost active user:", most_active_user)
print("Number of ratings by this user:", most_active_user_count)

print("\nMost-rated movie:", most_rated_movie_title)
print("Movie ID:", most_rated_movie_id)
print("Number of ratings:", most_rated_movie_count)

print("\nAverage rating:", round(average_rating, 2))

EXPLORATORY DATA ANALYSIS
----------------------------------------
Total number of ratings: 430
Number of unique users: 50
Number of unique movies: 24

Most active user: U016
Number of ratings by this user: 12

Most-rated movie: Nairobi Nights
Movie ID: M001
Number of ratings: 27

Average rating: 3.48


## 4. Creating the User-Item Interaction Matrix

In this step, I will create a user-item interaction matrix to show how users have rated different movies. Each row will represent a user, while each column will represent a movie. The cells will contain the ratings given by each user, and missing values will indicate movies that a user has not rated.


In [14]:
# Create the user-item interaction matrix
user_item_matrix = df_clean.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
)

# Display the first 10 users and first 10 movies
print("User-Item Interaction Matrix (First 10 Users and Movies):")
display(user_item_matrix.iloc[:10, :10])

User-Item Interaction Matrix (First 10 Users and Movies):


movie_id,M001,M002,M003,M004,M005,M006,M007,M008,M009,M010
user_id,,,,,,,,,,
U001,3.0,2.0,NaN,NaN,4.0,4.0,NaN,NaN,NaN,NaN
U002,3.0,NaN,NaN,5.0,NaN,NaN,3.0,NaN,NaN,NaN
U003,NaN,NaN,NaN,NaN,4.0,NaN,NaN,4.0,4.0,NaN
U004,4.0,NaN,NaN,NaN,4.0,3.0,NaN,NaN,NaN,NaN
U005,2.0,4.0,NaN,3.0,NaN,NaN,5.0,NaN,NaN,NaN
U006,3.0,3.0,NaN,5.0,4.0,NaN,NaN,3.0,NaN,NaN
U007,3.0,NaN,NaN,NaN,5.0,4.0,NaN,NaN,2.0,NaN
U008,3.0,2.0,NaN,NaN,3.0,4.0,5.0,NaN,NaN,NaN
U009,NaN,NaN,NaN,4.0,4.0,3.0,NaN,4.0,4.0,NaN


## 5. Calculating Sparsity

Sparsity measures how many entries in the user-item matrix are empty because users have not rated certain movies. A higher sparsity value means that fewer user-movie combinations have ratings. I will calculate sparsity using the total number of ratings divided by the total possible user-movie combinations, then subtract the result from 1.


In [15]:
# Calculate the number of observed ratings
observed_ratings = df_clean.shape[0]

# Calculate the total possible user-movie combinations
total_possible_ratings = unique_users * unique_movies

# Calculate sparsity
sparsity = 1 - (observed_ratings / total_possible_ratings)

# Display the results
print("Number of observed ratings:", observed_ratings)
print("Number of users:", unique_users)
print("Number of movies:", unique_movies)
print("Total possible user-movie combinations:", total_possible_ratings)

print("\nSparsity:", round(sparsity, 4))
print("Sparsity percentage:", round(sparsity * 100, 2), "%")

Number of observed ratings: 430
Number of users: 50
Number of movies: 24
Total possible user-movie combinations: 1200

Sparsity: 0.6417
Sparsity percentage: 64.17 %


### Interpretation of Sparsity

The sparsity is **64.17%**, which means that 64.17% of all possible user-movie combinations have no rating. Out of 1,200 possible combinations (50 users × 24 movies), only 430 contain ratings, while 770 are empty. This shows that most users have rated only a small part of the catalogue. High sparsity is common in recommendation data and makes it harder to recommend movies to users with little rating history.

## 6. Building a Popularity-Based Recommender

In this section, I will build a simple popularity-based recommendation system by calculating the average rating and number of ratings for each movie. To make the recommendations more reliable, I will only consider movies that have received at least 10 ratings. I will then sort the movies by their average ratings and select the top 10.

In [16]:
# Calculate the average rating and number of ratings for each movie
movie_popularity = df_clean.groupby(
    ['movie_id', 'movie_title']
).agg(
    average_rating=('rating', 'mean'),
    number_of_ratings=('rating', 'count')
).reset_index()

# Keep only movies with at least 10 ratings
popular_movies = movie_popularity[
    movie_popularity['number_of_ratings'] >= 10
].copy()

# Sort movies by average rating, then by number of ratings
popular_movies = popular_movies.sort_values(
    by=['average_rating', 'number_of_ratings'],
    ascending=[False, False]
)

# Select the top 10 movies
top_10_movies = popular_movies.head(10).copy()

# Round average ratings to 2 decimal places for display
top_10_movies['average_rating'] = top_10_movies['average_rating'].round(2)

# Display the recommendations
print("TOP 10 POPULARITY-BASED MOVIE RECOMMENDATIONS")
display(top_10_movies)

TOP 10 POPULARITY-BASED MOVIE RECOMMENDATIONS


,movie_id,movie_title,average_rating,number_of_ratings
20,M021,Market Day,4.50,14
12,M013,Midnight in Mombasa,4.35,17
13,M014,Code Breakers,4.21,14
6,M007,Hidden Lake,4.14,14
19,M020,Home Again,4.13,15
18,M019,Future Africa,4.07,14
4,M005,Mountain Echoes,4.00,26
5,M006,City of Tomorrow,3.93,27
3,M004,Coastal Dreams,3.77,22
11,M012,The Final Goal,3.75,16


## 7. Exporting the Top 10 Recommendations

After generating the recommendations, I will save the top 10 movies in a CSV file. This will make it easy to share the results and include them in my GitHub repository as part of the assignment submission.

In [17]:
# Save the top 10 recommendations to a CSV file
top_10_movies.to_csv('top_10_recommended_movies.csv', index=False)

print("Top 10 recommendations saved successfully!")

# Download the CSV file to my computer
from google.colab import files
files.download('top_10_recommended_movies.csv')

Top 10 recommendations saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Interpretation of Results

**Why the recommended movies appear at the top:** The recommended movies have the highest average ratings among the movies that received at least 10 ratings. Market Day is first with an average of 4.50 from 14 ratings, followed by Midnight in Mombasa with 4.35 from 17 ratings. The minimum of 10 ratings prevents movies with only one or two high ratings from reaching the top by chance.

**Advantage of popularity-based recommendation:** It is simple and needs no information about the user. It can recommend movies to a brand-new user who has no rating history, and it is easy to calculate and explain.

**Limitations:**
1. **No personalisation:** Every user gets the same list, whatever their own taste. A user who dislikes a top-rated movie's genre would still be recommended it.
2. **Bias against new or less-rated movies:** Movies with few ratings cannot enter the list, and highly rated ones keep getting recommended, so they collect even more ratings. New movies have little chance to be discovered.